# PROJECT : Amazon Food Reviews Sentiment Analysis Using NLP & Deep Learning

## 1.📌Project Summary

Title: Amazon Fine Food Reviews Sentiment Analysis

Objective: Build an end‑to‑end sentiment analysis system using NLP, TDM, ML, and Deep Learning.

Scope: Compare traditional ML model(Logistic Regression) with deep learning models (Dense NN, RNN, LSTM, BiLSTM).

Dataset: Amazon Fine Food Reviews (Text + Score → Sentiment).

## 2. 🎯 Business Problem

Context: Reviews are critical for e‑commerce platforms.

Problem Statement (refined): Customer reviews are often lengthy and unstructured, making it difficult for shoppers to quickly assess product quality. Sellers struggle to extract actionable insights from thousands of reviews, limiting their ability to improve products and services.

Business Value:

Customers → faster decisions with summarized sentiment.

Sellers → identify recurring issues and track sentiment trends.

Amazon → improved trust, engagement, and sales.

## 3. 📚 Import Libraries

In [50]:
# Standard libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# NLP
import string
string.punctuation
import nltk
from nltk.corpus import stopwords
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

# ML models
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

# Deep Learning
from tensorflow.keras.models import Sequential
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.layers import Flatten, Dense,Dropout, Embedding, SimpleRNN, LSTM, Bidirectional
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

# Evaluation
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report, confusion_matrix

In [21]:
import warnings
warnings.filterwarnings('ignore')

## 4. 📂DATASET:

In [22]:
df = pd.read_csv(r"D:\Downloads\AmazonFoodReviews.csv")

In [23]:
df = df.loc[:,['Text' , 'Score']]

In [24]:
df = df.rename(columns={'Text' :'X', 'Score':'Y'})

In [25]:
df.head(2)

,X,Y
0,I have bought several of the Vitality canned d...,5
1,Product arrived labeled as Jumbo Salted Peanut...,1


In [26]:
# 3 * 3 (negative = 1 ,2 --> 1 , neutral = 3 --> 2 , positive = 4,5--> 3)

In [27]:
df.Y = df.Y.replace({1:1,2:1,3:2,4:3,5:3})

## 5.🧹 Data Cleaning & Preprocessing

In [28]:
df['X'] = df['X'].str.lower()

In [29]:
df['X'] = df['X'].fillna('')

In [30]:
df['X'].isnull().sum()

np.int64(0)

In [31]:
l1 = stopwords.words('english') 
def text_process(mess):            ### creating a function
    """                                                        ## a docstring
    1. remove the punctuation
    2. remove the stopwords
    3. return the list of clean textwords
    
    """
    nopunc = [char for char in mess if char not in string.punctuation] # list comprehnsion
    nopunc = "".join(nopunc)
    
    return [ word for word in nopunc.split() if word not in l1]

## 6. Traditional ML with TDM (CountVectorizer)

In [32]:
cnt_vect = CountVectorizer(analyzer=text_process).fit(df['X'])

In [33]:
tdm = cnt_vect.transform(df['X'])

In [34]:
len(cnt_vect.vocabulary_)

240626

In [35]:
tdm.shape

(568454, 240626)

## 7. Train-Test Split

In [36]:
train_tdm, test_tdm, train_y, test_y = train_test_split(tdm, df.Y, test_size=.2)

In [37]:
train_y.shape

(454763,)

### Logistic Regression

In [38]:
model_lr = LogisticRegression(max_iter=100)
model_lr.fit(train_tdm, train_y)
pred = model_lr.predict(test_tdm)
tab = confusion_matrix(test_y, pred)
print(tab)
print(classification_report(test_y , pred))

[[11624   913  3754]
 [ 1539  2823  4111]
 [ 1993  1409 85525]]
              precision    recall  f1-score   support

           1       0.77      0.71      0.74     16291
           2       0.55      0.33      0.41      8473
           3       0.92      0.96      0.94     88927

    accuracy                           0.88    113691
   macro avg       0.74      0.67      0.70    113691
weighted avg       0.87      0.88      0.87    113691



### AmazonFoodReview_Using_Dense_SimpleRNN_LSTM_Bidirectionallstm

In [39]:
df = pd.read_csv(r"D:\Downloads\AmazonFoodReviews.csv")

In [40]:
df = df.loc[:,['Text' , 'Score']]

In [41]:
df = df.rename(columns={'Text' :'X', 'Score':'Y'})

In [42]:
df.head(2)

,X,Y
0,I have bought several of the Vitality canned d...,5
1,Product arrived labeled as Jumbo Salted Peanut...,1


In [43]:
df.Y = df.Y.replace({1:1,2:1,3:2,4:3,5:3})

In [44]:
df_X = df.iloc[:,0]
df_Y = df.iloc[:,1]

X_train, X_test, Y_train, Y_test = train_test_split(df_X, df_Y, test_size=.2)

In [45]:
Y_train = to_categorical(Y_train)

In [46]:
Y_train.shape

(454763, 4)

## 7. Deep Learning Models

### Tokenization + Padding

In [47]:
max_num_words = 20000
seq_len = 100
embedding_size = 100

tokenizer = Tokenizer(num_words = max_num_words)
tokenizer.fit_on_texts(df_X)

X_train = tokenizer.texts_to_sequences(X_train)
X_test = tokenizer.texts_to_sequences(X_test)

X_train = pad_sequences(X_train, maxlen = seq_len)
X_test = pad_sequences(X_test, maxlen = seq_len)

In [48]:
len(X_train[1000])

100

### Dense Neural Network

In [64]:
model_dense = Sequential()
model_dense.add(Embedding(input_dim = max_num_words, input_length = seq_len, output_dim = embedding_size))
model_dense.add(Flatten())
model_dense.add(Dense(128, activation='relu'))
model_dense.add(Dropout(0.5))
model_dense.add(Dense(4, activation='softmax'))

from tensorflow.keras.optimizers import Adam
adam = Adam(learning_rate = .001)

model_dense.compile(optimizer = adam, loss= 'categorical_crossentropy', metrics = ['accuracy'])
model_dense.fit(X_train, Y_train, epochs = 2 , batch_size= 50, validation_split = .2)

Epoch 1/2
7277/7277 ━━━━━━━━━━━━━━━━━━━━ 303s 41ms/step - accuracy: 0.8662 - loss: 0.3704 - val_accuracy: 0.8888 - val_loss: 0.3100
Epoch 2/2
7277/7277 ━━━━━━━━━━━━━━━━━━━━ 332s 46ms/step - accuracy: 0.9202 - loss: 0.2215 - val_accuracy: 0.8960 - val_loss: 0.3142


In [65]:
pred = model_dense.predict(X_test)
pred_cate = pred.argmax(axis = 1)

print(confusion_matrix(pred_cate , Y_test))

print(classification_report(pred_cate , Y_test))

3553/3553 ━━━━━━━━━━━━━━━━━━━━ 11s 3ms/step
[[13186  1732  2509]
 [  726  3783  1226]
 [ 2518  2967 85044]]
              precision    recall  f1-score   support

           1       0.80      0.76      0.78     17427
           2       0.45      0.66      0.53      5735
           3       0.96      0.94      0.95     90529

    accuracy                           0.90    113691
   macro avg       0.74      0.79      0.75    113691
weighted avg       0.91      0.90      0.90    113691



In [66]:
cm = confusion_matrix(pred_cate, Y_test)

accuracy = cm.diagonal().sum() / cm.sum()
print(f"Accuracy: {accuracy * 100:.2f}%")

Accuracy: 89.73%


### Simple RNN

In [52]:
model_rnn = Sequential()
model_rnn.add(Embedding(input_dim = max_num_words, input_length = seq_len, output_dim = embedding_size))
model_rnn.add(SimpleRNN(128))
model_rnn.add(Dense(4, activation= 'softmax'))

from tensorflow.keras.optimizers import Adam
adam = Adam(learning_rate = .001)

model_rnn.compile(optimizer = adam, loss= 'categorical_crossentropy', metrics = ['accuracy'])
model_rnn.fit(X_train, Y_train, epochs = 2 , batch_size= 50, validation_split = .2)

Epoch 1/2
7277/7277 ━━━━━━━━━━━━━━━━━━━━ 415s 57ms/step - accuracy: 0.8001 - loss: 0.5661 - val_accuracy: 0.8148 - val_loss: 0.5337
Epoch 2/2
7277/7277 ━━━━━━━━━━━━━━━━━━━━ 408s 56ms/step - accuracy: 0.8105 - loss: 0.5315 - val_accuracy: 0.8166 - val_loss: 0.5004


In [53]:
pred = model_rnn.predict(X_test)
pred_cate = pred.argmax(axis = 1)

print(confusion_matrix(pred_cate , Y_test))

print(classification_report(pred_cate , Y_test))

3553/3553 ━━━━━━━━━━━━━━━━━━━━ 40s 11ms/step
[[    0     1     0     2]
 [    0  5739  1019  1810]
 [    0    89   220   153]
 [    0 10601  7243 86814]]
              precision    recall  f1-score   support

           0       0.00      0.00      0.00         3
           1       0.35      0.67      0.46      8568
           2       0.03      0.48      0.05       462
           3       0.98      0.83      0.90    104658

    accuracy                           0.82    113691
   macro avg       0.34      0.49      0.35    113691
weighted avg       0.93      0.82      0.86    113691



In [54]:
cm = confusion_matrix(pred_cate, Y_test)

accuracy = cm.diagonal().sum() / cm.sum()
print(f"Accuracy: {accuracy * 100:.2f}%")

Accuracy: 81.60%


### LSTM

In [55]:
model_lstm = Sequential()
model_lstm.add(Embedding(input_dim = max_num_words, input_length = seq_len, output_dim = embedding_size))
model_lstm.add(LSTM(128))
model_lstm.add(Dense(4, activation= 'softmax'))

from tensorflow.keras.optimizers import Adam
adam = Adam(learning_rate = .001)

model_lstm.compile(optimizer = adam, loss= 'categorical_crossentropy', metrics = ['accuracy'])
model_lstm.fit(X_train, Y_train, epochs = 2 , batch_size= 50, validation_split = .2)

Epoch 1/2
7277/7277 ━━━━━━━━━━━━━━━━━━━━ 1044s 143ms/step - accuracy: 0.8690 - loss: 0.3547 - val_accuracy: 0.8855 - val_loss: 0.3053
Epoch 2/2
7277/7277 ━━━━━━━━━━━━━━━━━━━━ 969s 133ms/step - accuracy: 0.9103 - loss: 0.2453 - val_accuracy: 0.9046 - val_loss: 0.2711


In [56]:
pred = model_lstm.predict(X_test)
pred_cate = pred.argmax(axis = 1)

print(confusion_matrix(pred_cate , Y_test))

print(classification_report(pred_cate , Y_test))

3553/3553 ━━━━━━━━━━━━━━━━━━━━ 113s 32ms/step
[[12698  1531   878]
 [  757  2817   507]
 [ 2975  4134 87394]]
              precision    recall  f1-score   support

           1       0.77      0.84      0.81     15107
           2       0.33      0.69      0.45      4081
           3       0.98      0.92      0.95     94503

    accuracy                           0.91    113691
   macro avg       0.70      0.82      0.74    113691
weighted avg       0.93      0.91      0.92    113691



In [57]:
cm = confusion_matrix(pred_cate, Y_test)

accuracy = cm.diagonal().sum() / cm.sum()
print(f"Accuracy: {accuracy * 100:.2f}%")

Accuracy: 90.52%


### Bidirectional LSTM

In [58]:
model_bilstm = Sequential()
model_bilstm.add(Embedding(input_dim = max_num_words, input_length = seq_len, output_dim = embedding_size))
model_bilstm.add(Bidirectional(LSTM(128)))
model_bilstm.add(Dense(4, activation= 'softmax'))

from tensorflow.keras.optimizers import Adam
adam = Adam(learning_rate = .001)

model_bilstm.compile(optimizer = adam, loss= 'categorical_crossentropy', metrics = ['accuracy'])
model_bilstm.fit(X_train, Y_train, epochs = 2 , batch_size= 50, validation_split = .2)

Epoch 1/2
7277/7277 ━━━━━━━━━━━━━━━━━━━━ 1667s 228ms/step - accuracy: 0.8716 - loss: 0.3469 - val_accuracy: 0.8934 - val_loss: 0.3005
Epoch 2/2
7277/7277 ━━━━━━━━━━━━━━━━━━━━ 1491s 205ms/step - accuracy: 0.9118 - loss: 0.2416 - val_accuracy: 0.9040 - val_loss: 0.2685


In [59]:
pred = model_bilstm.predict(X_test)
pred_cate = pred.argmax(axis = 1)

print(confusion_matrix(pred_cate , Y_test))

print(classification_report(pred_cate , Y_test))

3553/3553 ━━━━━━━━━━━━━━━━━━━━ 156s 44ms/step
[[14073  2518  1920]
 [  713  3354  1488]
 [ 1644  2610 85371]]
              precision    recall  f1-score   support

           1       0.86      0.76      0.81     18511
           2       0.40      0.60      0.48      5555
           3       0.96      0.95      0.96     89625

    accuracy                           0.90    113691
   macro avg       0.74      0.77      0.75    113691
weighted avg       0.92      0.90      0.91    113691



In [60]:
cm = confusion_matrix(pred_cate, Y_test)

accuracy = cm.diagonal().sum() / cm.sum()
print(f"Accuracy: {accuracy * 100:.2f}%")

Accuracy: 90.42%


## 8. Evaluation

In [77]:
def evaluate_model(model, X_test, y_test):
    y_pred = model.predict(X_test)
    y_pred_classes = np.argmax(y_pred, axis=1)
    report_labels = [0, 1, 2, 3]
    target_names = ['Class 0 (unassigned)', 'Negative', 'Neutral', 'Positive']
    print(classification_report(y_test, y_pred_classes, labels=report_labels, target_names=target_names, zero_division=0))

In [72]:
# Example: Dense NN evaluation
print('Evaluation of the trained model:')
evaluate_model(model_dense, X_test, Y_test)

Evaluation of the trained model:
3553/3553 ━━━━━━━━━━━━━━━━━━━━ 10s 3ms/step
              precision    recall  f1-score   support

    Negative       0.76      0.80      0.78     16430
     Neutral       0.66      0.45      0.53      8482
    Positive       0.94      0.96      0.95     88779

    accuracy                           0.90    113691
   macro avg       0.79      0.74      0.75    113691
weighted avg       0.89      0.90      0.89    113691



In [78]:
# Example: Simple RNN evaluation
print('Evaluation of the trained model:')
evaluate_model(model_rnn, X_test, Y_test)

Evaluation of the trained model:
3553/3553 ━━━━━━━━━━━━━━━━━━━━ 52s 15ms/step
                      precision    recall  f1-score   support

Class 0 (unassigned)       0.00      0.00      0.00         0
            Negative       0.67      0.35      0.46     16430
             Neutral       0.48      0.03      0.05      8482
            Positive       0.83      0.98      0.90     88779

            accuracy                           0.82    113691
           macro avg       0.49      0.34      0.35    113691
        weighted avg       0.78      0.82      0.77    113691



In [76]:
# Example: LSTM evaluation
print('Evaluation of the trained model:')
evaluate_model(model_lstm, X_test, Y_test)

Evaluation of the trained model:
3553/3553 ━━━━━━━━━━━━━━━━━━━━ 110s 31ms/step
              precision    recall  f1-score   support

    Negative       0.84      0.77      0.81     16430
     Neutral       0.69      0.33      0.45      8482
    Positive       0.92      0.98      0.95     88779

    accuracy                           0.91    113691
   macro avg       0.82      0.70      0.74    113691
weighted avg       0.90      0.91      0.89    113691



In [79]:
# Example: Bidirectional LSTM evaluation
print('Evaluation of the trained model:')
evaluate_model(model_bilstm, X_test, Y_test)

Evaluation of the trained model:
3553/3553 ━━━━━━━━━━━━━━━━━━━━ 170s 48ms/step
                      precision    recall  f1-score   support

Class 0 (unassigned)       0.00      0.00      0.00         0
            Negative       0.76      0.86      0.81     16430
             Neutral       0.60      0.40      0.48      8482
            Positive       0.95      0.96      0.96     88779

            accuracy                           0.90    113691
           macro avg       0.58      0.55      0.56    113691
        weighted avg       0.90      0.90      0.90    113691



## 🔎 Key Observations

1. Logistic Regression

Achieved 88% accuracy and provides a strong traditional machine-learning baseline.
It performs reasonably well on Classes 1 and 3.
However, its recall for Class 2 is relatively low (0.33), indicating difficulty in identifying this class.

2. Dense Neural Network

Improved accuracy to 90%.
It achieved better recall for Class 2 (0.66) compared with Logistic Regression.
Its macro F1-score (0.75) shows more balanced performance across the classes.

3. Simple RNN

Achieved the lowest accuracy of 82%.
It performed poorly on Class 2, with an F1-score of only 0.05.
This suggests that the Simple RNN was not able to capture the contextual/sequential information in the reviews as effectively as LSTM-based models.

4. LSTM

Achieved the highest accuracy of 91%.
It obtained the highest macro recall (0.82) and the highest weighted F1-score (0.92).
Class 3 was classified particularly well, with an F1-score of 0.95.
Overall, LSTM provided the best combination of accuracy and class-level performance in this experiment.

5. Bidirectional LSTM

Achieved 90% accuracy, very close to LSTM.
It achieved a macro F1-score of 0.75 and weighted F1-score of 0.91.
It performed particularly well on Class 3, achieving an F1-score of 0.96.
However, it did not outperform the standard LSTM in overall accuracy.

## 🏆 Final Model Selection

Based on the overall evaluation, LSTM is selected as the final model for the Amazon Food Reviews sentiment classification task.

Although Dense Neural Network and Bidirectional LSTM achieved 90% accuracy, LSTM achieved the highest overall accuracy of 91%, along with the highest macro recall (0.82) and weighted F1-score (0.92).

Therefore, the LSTM model provides the best overall performance among the evaluated models for this dataset.

- Final Selected Model: LSTM
Accuracy: 91% | Macro Recall: 0.82 | Weighted F1-score: 0.92